# Fako Online - Full Pipeline Server

**Bark TTS + SadTalker + Easy-Wav2Lip**

This notebook runs all three models on a single Colab T4 GPU session.

### Instructions
1. Set runtime to **T4 GPU** (Runtime > Change runtime type)
2. Run all cells in order
3. Copy the ngrok URL to your `.env` file as `COLAB_API_URL`
4. Run CLI commands from your local machine

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
# Cell 2: Load models from Drive
# Account B models (shared via shortcut)
shared_b_dir = '/content/drive/MyDrive/AI_Avatar_Models'

# Create session directories
os.makedirs('/content/SadTalker/checkpoints', exist_ok=True)
os.makedirs('/content/Easy-Wav2Lip/checkpoints', exist_ok=True)
os.makedirs('/content/outputs', exist_ok=True)

# Copy SadTalker checkpoints
!cp -r {shared_b_dir}/SadTalker/* /content/SadTalker/checkpoints/ 2>/dev/null || true

# Copy Easy-Wav2Lip checkpoints
!cp -r {shared_b_dir}/Easy-Wav2Lip/* /content/Easy-Wav2Lip/checkpoints/ 2>/dev/null || true

# Set Bark cache to Drive
os.environ['SUNO_OFFLOAD_CPU'] = 'True'
os.environ['HF_HOME'] = f'{shared_b_dir}/Bark_TTS'

print('All models loaded from Drive!')

In [ ]:
# Cell 3: Install dependencies
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q transformers scipy
!pip install -q gfpgan
!pip install -q torch torchvision
!pip install -q librosa
print('Dependencies installed!')

In [ ]:
# Cell 4: Import libraries
import torch
import numpy as np
import io
import base64
import subprocess
import tempfile
from pathlib import Path

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 5: Load SadTalker
import sys
sys.path.insert(0, '/content/SadTalker')

from src.gradio_demo import SadTalker

sadtalker = SadTalker(
    checkpoint_path='/content/SadTalker/checkpoints',
    config_path='/content/SadTalker/src/config',
    lazy_dir='/content/SadTalker/gfpgan/weights'
)
print('SadTalker loaded!')

In [ ]:
# Cell 6: Load Easy-Wav2Lip
# We will import the inference module when needed
print('Easy-Wav2Lip ready to load on first use')

In [ ]:
# Cell 7: Load Bark TTS
from transformers import AutoProcessor, BarkModel

processor = AutoProcessor.from_pretrained('suno/bark')
model = BarkModel.from_pretrained('suno/bark')
model.to('cpu')  # Load to CPU first, move to GPU during inference

print('Bark TTS loaded!')

In [ ]:
# Cell 8: Define generation functions
import soundfile as sf

def generate_tts(text, voice_preset='v2/en_speaker_6'):
    """Generate emotional speech from text with Bark tags."""
    inputs = processor(text, voice_preset=voice_preset, return_tensors='pt')
    inputs = {k: v.to('cuda') if hasattr(v, 'to') else v for k, v in inputs.items()}
    model.to('cuda')
    
    with torch.no_grad():
        audio_values = model.generate(**inputs, do_sample=True)
    
    audio = audio_values.cpu().numpy().squeeze()
    model.to('cpu')
    torch.cuda.empty_cache()
    
    output_path = '/content/outputs/emotional_speech.wav'
    sf.write(output_path, audio, model.generation_config.sample_rate)
    return output_path

def generate_avatar(image_path, audio_path, result_filename='talking-head.mp4'):
    """Generate talking head video from image + audio."""
    result = sadtalker.test(
        image_path=image_path,
        audio_path=audio_path,
        result_dir='/content/outputs',
        still_mode=False,
        use_enhancer=True,
        batch_size=2,
        size=256,
        pose_style=0
    )
    return result['mp4_path'] if isinstance(result, dict) else result

def run_wav2lip(video_path, audio_path):
    """Refine lip sync with Easy-Wav2Lip."""
    output_path = '/content/outputs/refined_output.mp4'
    cmd = [
        'python', '/content/Easy-Wav2Lip/inference.py',
        '--checkpoint_path', '/content/Easy-Wav2Lip/checkpoints/wav2lip_gan.pth',
        '--face', video_path,
        '--audio', audio_path,
        '--outfile', output_path,
        '--nosmooth'
    ]
    subprocess.run(cmd, check=True, capture_output=True)
    return output_path

print('Generation functions defined!')

In [ ]:
# Cell 9: Create FastAPI server
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, JSONResponse
import uvicorn

app = FastAPI(title='Fako Online - Full Pipeline API')

@app.get('/health')
async def health():
    return {'status': 'ok', 'models': ['bark', 'sadtalker', 'wav2lip']}

@app.post('/generate-tts')
async def api_generate_tts(text: str = Form(...), voice_preset: str = Form('v2/en_speaker_6')):
    try:
        audio_path = generate_tts(text, voice_preset)
        return FileResponse(audio_path, media_type='audio/wav')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/generate-avatar')
async def api_generate_avatar(image: UploadFile = File(...), audio: UploadFile = File(...)):
    try:
        image_path = f'/content/outputs/{image.filename}'
        audio_path = f'/content/outputs/{audio.filename}'
        
        with open(image_path, 'wb') as f:
            f.write(await image.read())
        with open(audio_path, 'wb') as f:
            f.write(await audio.read())
        
        video_path = generate_avatar(image_path, audio_path)
        return FileResponse(video_path, media_type='video/mp4')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

@app.post('/generate-full')
async def api_generate_full(image: UploadFile = File(...), text: str = Form(...), voice_preset: str = Form('v2/en_speaker_6'), refine_lips: bool = Form(True)):
    try:
        image_path = f'/content/outputs/{image.filename}'
        with open(image_path, 'wb') as f:
            f.write(await image.read())
        
        audio_path = generate_tts(text, voice_preset)
        video_path = generate_avatar(image_path, audio_path)
        
        if refine_lips:
            video_path = run_wav2lip(video_path, audio_path)
        
        return FileResponse(video_path, media_type='video/mp4')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

print('FastAPI server defined!')

In [ ]:
# Cell 10: Start server with ngrok
from pyngrok import ngrok

# Kill any existing tunnels
ngrok.kill()

# Create tunnel
public_url = ngrok.bind(8000)
print(f'\nPublic URL: {public_url}')
print(f'\nUpdate your .env file:')
print(f'COLAB_API_URL={public_url}')

# Start the server
uvicorn.run(app, host='0.0.0.0', port=8000)